# Phase 0 — Foundations & Setup

**Goal:** Get the repo, Python environment, Ollama, and Hugging Face access ready so every later notebook just works.

By the end of this notebook you will have:

1. A working virtual environment with all dependencies installed.
2. Ollama running locally with at least one LLM and one VLM pulled.
3. A `sentence-transformers` embedding model cached locally.
4. A sanity-checked import of every key library.
5. A clear mental model of the repo layout and how the phases map to files.

> **Time:** ~30 minutes (plus model download time). No API keys required.


## 0.1 — The repo layout, briefly

```
multimodal-research-teaching-assistant/
├── app/                  # production code
│   ├── api/              # FastAPI
│   ├── core/             # config, llm client, rag pipeline
│   ├── ingestion/        # pdf_loader, chunker, figure_extractor
│   ├── retrieval/        # vector_store, embedder, reranker
│   ├── multimodal/       # CLIP, VLM client
│   ├── evaluation/       # Ragas / DeepEval wrappers
│   └── prompts/          # Jinja2 templates
├── frontend/             # Streamlit
├── data/                 # raw/processed/vector_store/logs (gitignored)
├── notebooks/            # this tutorial series
├── tests/                # pytest
├── docker/               # Dockerfile.api, Dockerfile.ui, compose
└── README.md
```

Each notebook in this series corresponds to one or two phases from the project plan, and each is fully runnable on local models — no API keys, no cloud spend.


## 0.2 — Create the virtual environment

Run these in your shell (not in the notebook):

```bash
cd multimodal-research-teaching-assistant
python3.14 -m venv .venv
source .venv/bin/activate            
# Windows: .venv\Scripts\activate
pip install --upgrade pip
pip install -r requirements.txt
python -m ipykernel install --user --name mmrta --display-name "Python (mmrta)"
```

Then in Jupyter, switch the kernel to **Python (mmrta)** so the notebook uses your venv.


In [ ]:
# Quick sanity check of the Python environment.
import sys, platform
print(f"Python      : {sys.version.split()[0]}")
print(f"Platform    : {platform.platform()}")
print(f"Executable  : {sys.executable}")


## 0.3 — Install Ollama and pull models

[Ollama](https://ollama.com) is the easiest way to run open-source LLMs and VLMs locally. Install it from the website, then in your shell:

```bash
ollama pull llama3.2:3b        # ~2 GB, our default LLM
ollama pull llava:7b           # ~4.7 GB, our default VLM (vision-language)
ollama pull nomic-embed-text   # optional alternative embedder, ~274 MB
```

Start the Ollama server (the installer usually does this automatically):

```bash
ollama serve   # if it is not already running
```

Verify it is up:


In [ ]:
import httpx

OLLAMA_HOST = "http://localhost:11434"

try:
    r = httpx.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
    r.raise_for_status()
    models = [m["name"] for m in r.json().get("models", [])]
    print("Ollama is running. Installed models:")
    for m in models:
        print("  -", m)
    if not any("llama3.2" in m for m in models):
        print("\n[warn] run: ollama pull llama3.2:3b")
    if not any("llava" in m for m in models):
        print("[warn] run: ollama pull llava:7b")
except Exception as e:
    print("Ollama not reachable. Make sure `ollama serve` is running.")
    print("   Error:", e)


## 0.4 — First call to a local LLM

We will use the official `ollama` Python client. This is the same call we will wrap in `app/core/llm.py` later.


In [ ]:
import ollama

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "system", "content": "You are a concise teaching assistant."},
        {"role": "user", "content": "In one sentence, what is retrieval-augmented generation?"},
    ],
    options={"temperature": 0.2},
)
print(response["message"]["content"])


## 0.5 — Cache the embedding model

`sentence-transformers/all-MiniLM-L6-v2` is small (~90 MB), CPU-fast, and good enough for paper-scale corpora. We will use it as our default text embedder.


In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
vec = embedder.encode(["Hello, attention is all you need."])
print("Embedding shape:", vec.shape, "  (n_sentences, dim)")
print("First 8 dims:   ", vec[0, :8])


## 0.6 — Confirm the key imports all work

If any of these fail, fix it now — every later notebook depends on them.


In [ ]:
import importlib

required = [
    "fitz",                   # PyMuPDF
    "pdfplumber",
    "PIL",                    # Pillow
    "sentence_transformers",
    "faiss",
    "ollama",
    "transformers",
    "torch",
    "open_clip",              # open_clip_torch
    "fastapi",
    "streamlit",
    "ragas",
    "langchain_text_splitters",
]
missing = []
for name in required:
    try:
        importlib.import_module(name)
        print(f"  ok  {name}")
    except ImportError as e:
        missing.append(name)
        print(f"  --  {name}  ({e})")

if missing:
    print("\nMissing modules — run: pip install -r requirements.txt")
else:
    print("\nEnvironment is ready.")


## 0.7 — Set up `.env` and load the typed settings

```bash
cp .env.example .env
```

The defaults work out of the box for an Ollama-only setup. Open `.env` and look at the keys you can tune:

- `LLM_PROVIDER=ollama` — swap to `huggingface`, `openai`, `anthropic`, or `google` later.
- `OLLAMA_LLM_MODEL=llama3.2:3b` and `OLLAMA_VLM_MODEL=llava:7b`.
- `EMBEDDING_MODEL`, `CLIP_MODEL`.
- `VECTOR_STORE=faiss` — swap to `qdrant` in Notebook 09's deployment section.
- `TOP_K`, `CHUNK_SIZE`, `CHUNK_OVERLAP`.


In [ ]:
# This works if you run the notebook from the repo root.
import sys, os
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "app").is_dir() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))
os.chdir(repo_root)

from app.core.config import settings

print("LLM provider :", settings.llm_provider)
print("LLM model    :", settings.ollama_llm_model)
print("VLM model    :", settings.ollama_vlm_model)
print("Embedder     :", settings.embedding_model)
print("Vector store :", settings.vector_store)
print("Top-K        :", settings.top_k)


## What you learned

- The repo layout and how phases map to `app/` subpackages.
- How to set up a venv, install dependencies, and register a Jupyter kernel.
- How to install Ollama and pull `llama3.2:3b` + `llava:7b`.
- How to call the local LLM from Python.
- How to load a sentence-transformer embedder and run an embedding.
- How to load the typed settings object from `app/core/config.py`.

## Exercises

1. Pull a second LLM (e.g. `mistral:7b`) and benchmark its latency against `llama3.2:3b` on the same prompt.
2. Run the embedder on a batch of 100 sentences and measure throughput (sentences/sec).
3. Edit `.env` to change `TOP_K` to 8 and re-run the settings cell — confirm the value updates.

## Next: [Phase 1 — PDF ingestion](./2026-05-25-phase01-pdf-ingestion.ipynb)
